<a href="https://colab.research.google.com/github/Cata-X/GenIA/blob/main/Codigo_Modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch bitsandbytes accelerate huggingface_hub

from huggingface_hub import notebook_login
# Pega tu Access Token de Hugging Face cuando te lo pida
notebook_login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.9 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 2. Configuración de modelo e instanciación (Menlo/AlphaMaze-v0.2-1.5B)
model_id = "Menlo/AlphaMaze-v0.2-1.5B"

# Configuración de cuantización a 4 bits para la GPU T4 (~5.5 GB VRAM)
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

print("Cargando AlphaMaze-v0.2-1.5B...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

# 3. Definición del laberinto en MATRIZ DE 9x9 (0 = libre, 1 = muralla)
laberinto = [
    [0, 0, 1, 0, 0, 0, 0, 0, 0], # Fila 0 -> Inicio en (0,0) y Meta en (0,8)
    [1, 0, 1, 0, 1, 1, 1, 1, 0], # Fila 1
    [0, 0, 0, 0, 1, 0, 0, 1, 0], # Fila 2
    [0, 1, 1, 1, 1, 0, 1, 1, 0], # Fila 3
    [0, 0, 0, 0, 0, 0, 0, 1, 0], # Fila 4
    [1, 1, 1, 1, 1, 1, 0, 1, 0], # Fila 5
    [0, 0, 0, 0, 0, 1, 0, 0, 0], # Fila 6
    [0, 1, 1, 1, 0, 1, 1, 1, 1], # Fila 7
    [0, 0, 0, 0, 0, 0, 0, 0, 0]  # Fila 8
]

filas = len(laberinto)
columnas = len(laberinto[0])
matriz_str = "\n".join([str(fila) for fila in laberinto])

prompt_diagnostico = f"""
Dada la siguiente matriz de un laberinto de TAMAÑO ESPECÍFICO {filas} filas por {columnas} columnas:

{matriz_str}

RESTRICCIONES DE DIMENSIÓN Y MAPA:
- El mapa tiene exactamente {filas} filas (índices 0 a {filas-1}) y {columnas} columnas (índices 0 a {columnas-1}).
- '0' es un camino libre y '1' es una muralla infranqueable.
- Inicio: (0, 0) [Fila 0, Columna 0]
- Meta: (0, {columnas-1}) [Fila 0, Columna {columnas-1}]

REGLA ESTRICTA DE DIAGNÓSTICO:
Propón una ruta para llegar a la meta. Para CADA PASO que escribas, debes incluir obligatoriamente:
1. La coordenada propuesta: (Fila, Columna).
2. El valor real en la matriz para esa coordenada (¿Es 0 o 1?).
3. Verificación de adyacencia: Confirma que el movimiento es exactamente a 1 casilla de distancia adyacente (Arriba, Abajo, Izquierda o Derecha) desde la coordenada anterior.

Escribe la solución paso a paso.
"""

# 4. Formatear prompt con la plantilla oficial del chat y enviar a la GPU
messages = [{"role": "user", "content": prompt_diagnostico}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

# 5. Generación de respuesta
print("\nEjecutando inferencia en matriz 9x9...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1000, # Incrementado ligeramente para permitir más pasos si los genera
        temperature=0.1,    # Temperatura baja para forzar seguimiento estricto
        do_sample=False
    )

# 6. Decodificar e imprimir el diagnóstico
respuesta = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("--- DIAGNÓSTICO DE INFERENCIA DIRECTA (MATRIZ 9x9) ---")
print(respuesta)

Cargando Llama-3.1-8B-Instruct...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Ejecutando inferencia en matriz 9x9...
--- DIAGNÓSTICO DE INFERENCIA DIRECTA (MATRIZ 9x9) ---
<think>
Primero, noto que el problema se encanta de una matriz 9x9, donde 0 es un camino libre y 1 es una muralla. El camino inicia en (0,0) y termina en (0,8). 

Para intentar encontrar una ruta, haremos pasos por pasos, siguiendo las reglas strictas:

1. **Primer paso**: En (0,0), el valor es 0, lo que significa que no hay muralla. Esto tells que podemos mover a la derecha o a la arriba, pero no a la izquierda o a la abajo.

2. **Segundo paso**: En (0,1), el valor es 0, lo que también significa que no hay muralla. Por lo tanto, podemos mover a la derecha o a la arriba.

3. **Tercer paso**: En (0,2), el valor es 1, lo que significa que no podemos mover a la izquierda. Por lo tanto, tenemos dos opciones: (0,3) o (1,2).

4. **Cuarto paso**: En (0,3), el valor es 0, lo que significa que no hay muralla. Por lo tanto, podemos mover a la derecha o a la arriba.

5. **C Quinto paso**: En (0,4), el v

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 2. Configuración de modelo e instanciación (Menlo/AlphaMaze-v0.2-1.5B)
model_id = "meta-llama/Llama-3.1-8B-Instruct"

# Configuración de cuantización a 4 bits para la GPU T4 (~5.5 GB VRAM)
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

print("Cargando Llama-3.1-8B-Instruct...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

# 3. Definición del laberinto en MATRIZ DE 9x9 (0 = libre, 1 = muralla)
laberinto = [
    [0, 0, 1, 0, 0, 0, 0, 0, 0], # Fila 0 -> Inicio en (0,0) y Meta en (0,8)
    [1, 0, 1, 0, 1, 1, 1, 1, 0], # Fila 1
    [0, 0, 0, 0, 1, 0, 0, 1, 0], # Fila 2
    [0, 1, 1, 1, 1, 0, 1, 1, 0], # Fila 3
    [0, 0, 0, 0, 0, 0, 0, 1, 0], # Fila 4
    [1, 1, 1, 1, 1, 1, 0, 1, 0], # Fila 5
    [0, 0, 0, 0, 0, 1, 0, 0, 0], # Fila 6
    [0, 1, 1, 1, 0, 1, 1, 1, 1], # Fila 7
    [0, 0, 0, 0, 0, 0, 0, 0, 0]  # Fila 8
]

filas = len(laberinto)
columnas = len(laberinto[0])
matriz_str = "\n".join([str(fila) for fila in laberinto])

prompt_diagnostico = f"""
Dada la siguiente matriz de un laberinto de TAMAÑO ESPECÍFICO {filas} filas por {columnas} columnas:

{matriz_str}

RESTRICCIONES DE DIMENSIÓN Y MAPA:
- El mapa tiene exactamente {filas} filas (índices 0 a {filas-1}) y {columnas} columnas (índices 0 a {columnas-1}).
- '0' es un camino libre y '1' es una muralla infranqueable.
- Inicio: (0, 0) [Fila 0, Columna 0]
- Meta: (0, {columnas-1}) [Fila 0, Columna {columnas-1}]

REGLA ESTRICTA DE DIAGNÓSTICO:
Propón una ruta para llegar a la meta. Para CADA PASO que escribas, debes incluir obligatoriamente:
1. La coordenada propuesta: (Fila, Columna).
2. El valor real en la matriz para esa coordenada (¿Es 0 o 1?).
3. Verificación de adyacencia: Confirma que el movimiento es exactamente a 1 casilla de distancia adyacente (Arriba, Abajo, Izquierda o Derecha) desde la coordenada anterior.

Escribe la solución paso a paso.
"""

# 4. Formatear prompt con la plantilla oficial del chat y enviar a la GPU
messages = [{"role": "user", "content": prompt_diagnostico}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

# 5. Generación de respuesta
print("\nEjecutando inferencia en matriz 9x9...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1000, # Incrementado ligeramente para permitir más pasos si los genera
        temperature=0.1,    # Temperatura baja para forzar seguimiento estricto
        do_sample=False
    )

# 6. Decodificar e imprimir el diagnóstico
respuesta = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("--- DIAGNÓSTICO DE INFERENCIA DIRECTA (MATRIZ 9x9) ---")
print(respuesta)

Cargando Llama-3.1-8B-Instruct...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


Ejecutando inferencia en matriz 9x9...
--- DIAGNÓSTICO DE INFERENCIA DIRECTA (MATRIZ 9x9) ---
Para encontrar la ruta más corta desde el punto de inicio (0, 0) hasta la meta (0, 8), podemos utilizar un algoritmo de búsqueda en profundidad (DFS) o un algoritmo de búsqueda en anchura (BFS). En este caso, utilizaremos el algoritmo de BFS.

**Paso 1:**
Coordenada propuesta: (0, 1)
Valor real en la matriz: 1
Verificación de adyacencia: El movimiento es exactamente a 1 casilla de distancia adyacente (izquierda) desde la coordenada anterior (0, 0).

**Paso 2:**
Coordenada propuesta: (0, 2)
Valor real en la matriz: 1
Verificación de adyacencia: El movimiento es exactamente a 1 casilla de distancia adyacente (izquierda) desde la coordenada anterior (0, 1).

**Paso 3:**
Coordenada propuesta: (0, 3)
Valor real en la matriz: 0
Verificación de adyacencia: El movimiento es exactamente a 1 casilla de distancia adyacente (izquierda) desde la coordenada anterior (0, 2).

**Paso 4:**
Coordenada propuest

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print("Cargando tokenizador y modelo...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("¡Modelo cargado exitosamente en la GPU!")

# 3. Definición del laberinto en MATRIZ DE 9x9 (0 = libre, 1 = muralla)
laberinto = [
    [0, 0, 1, 0, 0, 0, 0, 0, 0], # Fila 0 -> Inicio en (0,0) y Meta en (0,8)
    [1, 0, 1, 0, 1, 1, 1, 1, 0], # Fila 1
    [0, 0, 0, 0, 1, 0, 0, 1, 0], # Fila 2
    [0, 1, 1, 1, 1, 0, 1, 1, 0], # Fila 3
    [0, 0, 0, 0, 0, 0, 0, 1, 0], # Fila 4
    [1, 1, 1, 1, 1, 1, 0, 1, 0], # Fila 5
    [0, 0, 0, 0, 0, 1, 0, 0, 0], # Fila 6
    [0, 1, 1, 1, 0, 1, 1, 1, 1], # Fila 7
    [0, 0, 0, 0, 0, 0, 0, 0, 0]  # Fila 8
]

filas = len(laberinto)
columnas = len(laberinto[0])
matriz_str = "\n".join([str(fila) for fila in laberinto])

prompt_diagnostico = f"""
Dada la siguiente matriz de un laberinto de TAMAÑO ESPECÍFICO {filas} filas por {columnas} columnas:

{matriz_str}

RESTRICCIONES DE DIMENSIÓN Y MAPA:
- El mapa tiene exactamente {filas} filas (índices 0 a {filas-1}) y {columnas} columnas (índices 0 a {columnas-1}).
- '0' es un camino libre y '1' es una muralla infranqueable.
- Inicio: (0, 0) [Fila 0, Columna 0]
- Meta: (0, {columnas-1}) [Fila 0, Columna {columnas-1}]

REGLA ESTRICTA DE DIAGNÓSTICO:
Propón una ruta para llegar a la meta. Para CADA PASO que escribas, debes incluir obligatoriamente:
1. La coordenada propuesta: (Fila, Columna).
2. El valor real en la matriz para esa coordenada (¿Es 0 o 1?).
3. Verificación de adyacencia: Confirma que el movimiento es exactamente a 1 casilla de distancia adyacente (Arriba, Abajo, Izquierda o Derecha) desde la coordenada anterior.

Escribe la solución paso a paso.
"""

# 4. Formatear prompt con la plantilla oficial del chat y enviar a la GPU
messages = [{"role": "user", "content": prompt_diagnostico}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

# 5. Generación de respuesta
print("\nEjecutando inferencia en matriz 9x9...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1000, # Incrementado ligeramente para permitir más pasos si los genera
        temperature=0.1,    # Temperatura baja para forzar seguimiento estricto
        do_sample=False
    )

# 6. Decodificar e imprimir el diagnóstico
respuesta = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("--- DIAGNÓSTICO DE INFERENCIA DIRECTA (MATRIZ 9x9) ---")
print(respuesta)

Cargando tokenizador y modelo...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

¡Modelo cargado exitosamente en la GPU!

Ejecutando inferencia en matriz 9x9...
--- DIAGNÓSTICO DE INFERENCIA DIRECTA (MATRIZ 9x9) ---
### Paso 1: Coordenadas Inicial y Valor Matriz
**Coordenada:** (0, 0)
**Valor en Matriz:** 0

### Paso 2: Movimiento hacia arriba
**Coordenada:** (0, 1)
**Valor en Matriz:** 0

### Paso 3: Movimiento hacia derecha
**Coordenada:** (1, 1)
**Valor en Matriz:** 0

### Paso 4: Movimiento hacia abajo
**Coordenada:** (1, 2)
**Valor en Matriz:** 0

### Paso 5: Movimiento hacia izquierda
**Coordenada:** (0, 2)
**Valor en Matriz:** 0

### Paso 6: Movimiento hacia arriba
**Coordenada:** (0, 3)
**Valor en Matriz:** 0

### Paso 7: Movimiento hacia derecha
**Coordenada:** (1, 3)
**Valor en Matriz:** 0

### Paso 8: Movimiento hacia abajo
**Coordenada:** (1, 4)
**Valor en Matriz:** 0

### Paso 9: Movimiento hacia izquierda
**Coordenada:** (0, 4)
**Valor en Matriz:** 0

### Paso 10: Movimiento hacia arriba
**Coordenada:** (0, 5)
**Valor en Matriz:** 0

### Paso 11: Movi

In [ ]:
import gc
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. Definición del dataset de prueba (Matriz 9x9)
laberinto_9x9 = [
    [0, 0, 1, 0, 0, 0, 0, 0, 0],  # Inicio (0,0), Meta (0,8)
    [1, 0, 1, 0, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 1, 0, 0, 1, 0],
    [0, 1, 1, 1, 1, 0, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 0],
    [1, 1, 1, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 0, 1, 1, 1, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 0]
]

FILAS = len(laberinto_9x9)
COLUMNAS = len(laberinto_9x9[0])
INICIO = (0, 0)
META = (0, COLUMNAS - 1)
MATRIZ_STR = "\n".join([str(fila) for fila in laberinto_9x9])

# Lista de tus 3 modelos candidatos
modelos = [
    "Menlo/AlphaMaze-v0.2-1.5B",
    "meta-llama/Llama-3.1-8B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct"
]

prompt_base = f"""
Dada la matriz de laberinto de {FILAS}x{COLUMNAS} (0=camino libre, 1=muralla):
{MATRIZ_STR}

Inicio: {INICIO}
Meta: {META}

TAREA: Entrega únicamente la secuencia de coordenadas (Fila, Columna) desde el inicio hasta la meta.
Ejemplo de formato: (0,0) -> (0,1) -> (1,1) -> ...
"""

# 2. Función evaluadora determinista
def evaluar_camino(texto_respuesta, matriz, meta):
    # Extraer todas las tuplas de números (fila, col) del texto
    coordenadas = re.findall(r'\((\d+),\s*(\d+)\)', texto_respuesta)
    ruta = [(int(r), int(c)) for r, c in coordenadas]

    if not ruta:
        return {"exito": False, "colisiones": 0, "pasos": 0, "error": "Sin coordenadas extraídas"}

    colisiones = 0
    pasos_invalidos = 0

    for i, (f, c) in enumerate(ruta):
        # Validar límites de la matriz
        if f >= len(matriz) or c >= len(matriz[0]):
            pasos_invalidos += 1
            continue
        # Contar colisiones contra murallas
        if matriz[f][c] == 1:
            colisiones += 1

    llego_a_meta = (ruta[-1] == meta) if ruta else False

    return {
        "exito": llego_a_meta,
        "colisiones": colisiones,
        "pasos_totales": len(ruta),
        "coordenadas_generadas": ruta
    }

# 3. Bucle de ejecución para comparar los 3 modelos
resultados = {}
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

for model_id in modelos:
    print(f"\n==========================================")
    print(f"EVALUANDO MODELO: {model_id}")
    print(f"==========================================")

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=quantization_config,
            device_map="auto"
        )

        messages = [{"role": "user", "content": prompt_base}]
        formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=400,
                temperature=0.1,
                do_sample=False
            )

        respuesta_texto = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

        # Evaluar la respuesta del modelo
        metrics = evaluar_camino(respuesta_texto, laberinto_9x9, META)
        resultados[model_id] = metrics

        print(f"Respuesta cruda (primeros 150 caracteres):\n{respuesta_texto[:150]}...\n")
        print(f"Resultados de Evaluación: {metrics}")

    except Exception as e:
        print(f"Error al procesar el modelo {model_id}: {e}")

    # Limpieza estricta de memoria GPU VRAM antes de cargar el siguiente modelo
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

# 4. Tabla Resumen
print("\n" + "="*60)
print("TABLA COMPARATIVA FINAL DE BENCHMARK ESPACIAL")
print("="*60)
for mod, res in resultados.items():
    print(f"Modelo: {mod}")
    print(f"  - ¿Llegó a la Meta?: {res.get('exito')}")
    print(f"  - Colisiones con Murallas (1): {res.get('colisiones')}")
    print(f"  - Pasos Totales Propuestos: {res.get('pasos_totales')}")
    print("-" * 60)


EVALUANDO MODELO: Menlo/AlphaMaze-v0.2-1.5B


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Respuesta cruda (primeros 150 caracteres):
<think>
Primero, identifico la posición inicial del camino, que es (0, 0), y la posición fina, que es (0, 8).

Luego, trataré de seguir un camino cons...

Resultados de Evaluación: {'exito': True, 'colisiones': 1, 'pasos_totales': 11, 'coordenadas_generadas': [(0, 0), (0, 8), (0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8)]}

EVALUANDO MODELO: meta-llama/Llama-3.1-8B-Instruct


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Respuesta cruda (primeros 150 caracteres):
Para resolver este problema, podemos utilizar un algoritmo de búsqueda en profundidad (DFS) o un algoritmo de búsqueda en anchura (BFS). Aquí te prese...

Resultados de Evaluación: {'exito': False, 'colisiones': 1, 'pasos_totales': 2, 'coordenadas_generadas': [(0, 1), (1, 0)]}

EVALUANDO MODELO: Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Respuesta cruda (primeros 150 caracteres):
Para resolver este problema, podemos utilizar un algoritmo de búsqueda en profundidad o una solución similar para encontrar la ruta más corta desde el...

Resultados de Evaluación: {'exito': False, 'colisiones': 0, 'pasos_totales': 6, 'coordenadas_generadas': [(1, 1), (1, 1), (0, 0), (0, 1), (0, 1), (0, 0)]}

TABLA COMPARATIVA FINAL DE BENCHMARK ESPACIAL
Modelo: Menlo/AlphaMaze-v0.2-1.5B
  - ¿Llegó a la Meta?: True
  - Colisiones con Murallas (1): 1
  - Pasos Totales Propuestos: 11
------------------------------------------------------------
Modelo: meta-llama/Llama-3.1-8B-Instruct
  - ¿Llegó a la Meta?: False
  - Colisiones con Murallas (1): 1
  - Pasos Totales Propuestos: 2
------------------------------------------------------------
Modelo: Qwen/Qwen2.5-1.5B-Instruct
  - ¿Llegó a la Meta?: False
  - Colisiones con Murallas (1): 0
  - Pasos Totales Propuestos: 6
------------------------------------------------------------
